# Modulation

> Bits to symbols, and back.

In [ ]:
#| default_exp dsp.modulation

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from abc import ABC, abstractmethod

import numpy as np

from comm_core.block import _is_torch

In [ ]:
#| export
def _conj(h):
    return h.conj() if _is_torch(h) else np.conj(h)


def _real(z):
    if _is_torch(z):
        return z.real if z.is_complex() else z
    return np.real(z)

## Modulation

A `Modulation` maps bits to symbols with average energy 1. It detects them again knowing the channel coefficient
`h` (coherent detection, with perfect channel knowledge at the receiver).

In [ ]:
#| export
class Modulation(ABC):
    "Bits <-> symbols."

    bits_per_symbol: int = 1

    @abstractmethod
    def modulate(self, bits):
        "Symbols (..., num_bits / bits_per_symbol), with average energy 1."

    @abstractmethod
    def demodulate(self, received, h):
        "The detected bits (booleans) of the received symbols, knowing the channel coefficients `h`."


class BPSK(Modulation):
    "Binary phase-shift keying: bit 0 -> -1, bit 1 -> +1."

    bits_per_symbol = 1

    def modulate(self, bits):
        return 2.0 * bits - 1.0

    def demodulate(self, received, h):
        "Matched filter, which is the maximum-likelihood detector for BPSK: the sign of Re(y h*)."
        return _real(received * _conj(h)) > 0

In [ ]:
show_doc(BPSK.demodulate)

In [ ]:
bpsk = BPSK()
b = np.array([0, 1, 1, 0])
assert bpsk.modulate(b).tolist() == [-1, 1, 1, -1]
h = np.exp(1j * 2.0)                                               # any phase: the receiver knows h
assert (bpsk.demodulate(h * bpsk.modulate(b), h) == b).all()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()